# Data Note

From EDA, we found that data have some missing:
<ul>
    <li>EL       60
    <li>BQ       60
    <li>CC        3
    <li>FS        2
    <li>CB        2
    <li>FL        1
    <li>FC        1
    <li>DU        1
    <li>GL        1
</ul>

"EL" and "BQ" accounted around 10% of training data, so we have some treatment with it, others just a little bit, so just impute the data

# Load data

In [31]:
#-------------------------------------------------------------------------
# Import libraries
#-------------------------------------------------------------------------

import      os
import      pandas      as      pd
import      numpy       as      np

from        sklearn.ensemble    import  RandomForestRegressor
from        sklearn.metrics     import  mean_squared_error


#-------------------------------------------------------------------------
# Define path
#-------------------------------------------------------------------------
DATA_DIR    =   './data/'
train_path  =   os.path.join(DATA_DIR, 'train.csv')
test_path   =   os.path.join(DATA_DIR, 'test.csv')

In [32]:
#-------------------------------------------------------------------------
# Read data
#-------------------------------------------------------------------------

df_orin     =   pd.read_csv(train_path)
df_test     =   pd.read_csv(test_path)

In [14]:
df_orin.shape

(617, 58)

# Ver1: Not care about correlation, use all columns

## Impute low-missing columns

Note that all the low-missing columns is numericals type

In [39]:
#-------------------------------------------------------------------------
# Declare low-missing columns
#-------------------------------------------------------------------------
LOW_MISSING_COLUMNS =   ['CC', 'FS', 'CB', 'FL', 'FC', 'DU', 'GL']


#-------------------------------------------------------------------------
# For loop to impute
#-------------------------------------------------------------------------
for col in LOW_MISSING_COLUMNS:
    df_orin[col].fillna(df_orin[col].mean(), inplace=True)

In [40]:
#-------------------------------------------------------------------------
# Check again
#-------------------------------------------------------------------------
df_orin.columns[df_orin.isnull().sum()>0]

Index(['BQ', 'EL'], dtype='object')

## Approach 1: Just remove the row with missing

In [20]:
df_1 = df_orin.dropna()

In [21]:
#-------------------------------------------------------------------------
# Check again
#-------------------------------------------------------------------------
df_1.columns[df_1.isnull().sum()>0]

Index([], dtype='object')

In [27]:
print('Remain rows:', df_1.shape[0])
print('Account {}% of origin dataframe'.format(df_1.shape[0]/df_orin.shape[0]*100))

Remain rows: 550
Account 89.14100486223663% of origin dataframe


In [22]:
#-------------------------------------------------------------------------
# Save to dataframe
#-------------------------------------------------------------------------
df_1.to_csv(os.path.join(DATA_DIR, 'train_v1_1.csv'))

## Approach 2: Simple impute the missing row

Note that 'BQ' and 'EL' is numerical data type, so just impute by mean

In [23]:
df_2 = df_orin.copy()
for col in ['BQ', 'EL']:
    df_2[col].fillna(df_2[col].mean(), inplace=True)

#-------------------------------------------------------------------------
# Check again
#-------------------------------------------------------------------------
df_2.columns[df_2.isnull().sum()>0]

Index([], dtype='object')

In [24]:
#-------------------------------------------------------------------------
# Save to dataframe
#-------------------------------------------------------------------------
df_2.to_csv(os.path.join(DATA_DIR, 'train_v2_2.csv'))

## Approach 3: Estimate the columns value from other columns

In [45]:
def fill_missing(df, df_test):
    """
        Fill missing columns 'BQ' and 'EL' with random forest
    """
    COL1                =   'BQ'
    COL2                =   'EL'
    CATEGORICAL_COLUMNS =   ["Id", "EJ", "Class"]
    df                  =   df.drop(columns = CATEGORICAL_COLUMNS)
    df_test             =   df_test.drop(columns = ["Id", "EJ"])


    #---------------------------------------------------------------------
    # Step 1: Split the DataFrame into two parts: one with missing 
    # values and one without missing values
    #---------------------------------------------------------------------
    df_missing          =   df[df[COL1].isnull() & df[COL2].isnull()]
    df_no_missing       =   df.dropna(subset=[COL1, COL2])


    #---------------------------------------------------------------------
    # Step 2: Prepare the training data and labels
    #---------------------------------------------------------------------
    X_train             =   df_no_missing.drop(columns=[COL1, COL2])
    y_train_col1        =   df_no_missing[COL1]
    y_train_col2        =   df_no_missing[COL2]


    #---------------------------------------------------------------------
    # Step 3: Build separate models for 'col1' and 'col2'
    #---------------------------------------------------------------------
    model_col1          =   RandomForestRegressor()
    model_col2          =   RandomForestRegressor()

    # Train the models
    model_col1          .fit(X_train, y_train_col1)
    model_col2          .fit(X_train, y_train_col2)


    #---------------------------------------------------------------------
    # Step 4: Use the trained models to predict missing values
    #---------------------------------------------------------------------
    X_missing           =   df_missing.drop(columns=[COL1, COL2])
    df_missing[COL1]    =   model_col1.predict(X_missing)
    df_missing[COL2]    =   model_col2.predict(X_missing)

    # Combine the DataFrame with missing values and the DataFrame without missing values
    filled_df           =   pd.concat([df_no_missing, df_missing])


    #---------------------------------------------------------------------
    # Evaluate the model's performance
    # Assuming you have a true DataFrame with the actual values for 'col1' and 'col2' (df_true)
    #---------------------------------------------------------------------
    y_true_col1         =   df_test[COL1]
    y_true_col2         =   df_test[COL2]
    y_pred_col1         =   model_col1.predict(df_test.drop(columns=[COL1, COL2]))
    y_pred_col2         =   model_col2.predict(df_test.drop(columns=[COL1, COL2]))

    # Calculate evaluation metrics
    mse_col1            =   mean_squared_error(y_true_col1, y_pred_col1)
    mse_col2            =   mean_squared_error(y_true_col2, y_pred_col2)

    print("Mean Squared Error for col1:", mse_col1)
    print("Mean Squared Error for col2:", mse_col2)

    return filled_df

In [47]:
df_3 = fill_missing(df_orin, df_test)

Mean Squared Error for col1: 33562.110587644725
Mean Squared Error for col2: 4416.476005187956


/tmp/ipykernel_43234/4195365601.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_missing[COL1]    =   model_col1.predict(X_missing)
/tmp/ipykernel_43234/4195365601.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_missing[COL2]    =   model_col2.predict(X_missing)


In [48]:
#-------------------------------------------------------------------------
# Save to dataframe
#-------------------------------------------------------------------------
df_3.to_csv(os.path.join(DATA_DIR, 'train_v3_3.csv'))